# W04-1 주소에서 건물 외곽선까지

<a href="https://colab.research.google.com/github/kloud80/dataminig_cau/blob/main/%EC%A4%91%EC%95%99%EB%8C%8026%ED%8A%B9%EA%B0%95/notebooks/w04_colab/W04_1%20%EC%A3%BC%EC%86%8C%EC%97%90%EC%84%9C%20%EA%B1%B4%EB%AC%BC%20%EC%99%B8%EA%B3%BD%EC%84%A0%EA%B9%8C%EC%A7%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**주소 한 줄에서 3D 건물까지** — 그 첫 구간이다.
`서울특별시 강남구 영동대로 520` 이라는 글자에서 출발해 **건물 한 동의 바닥 윤곽(footprint)** 을 미터 단위로 손에 쥐는 데까지 간다.

이 노트북은 사내 연구 파이프라인 **addr2blend** 의 앞 단계를 단순하게 다시 짠 것이다.
파이프라인은 주소를 받아 공공데이터로 형상을 만들고, 거리 사진으로 입면을 판독해 Blender 모델을 만든다.
그 과정에서 **3D 형상은 사진이 아니라 공공데이터의 외곽선과 높이에서 나온다.** 오늘은 그 외곽선을 얻는다.

| 단계 | 하는 일 | 파이프라인 스테이지 |
|---|---|---|
| 1 | 주소 문자열 정리 | `s00_query` |
| 2 | 주소 → 경위도 (VWorld, 없으면 Nominatim) | `s01_candidates` 앞부분 |
| 3 | 경위도 → 국소 미터 좌표 (LOCAL_ENU) | `core/geo.py` |
| 4 | 주변 건물 윤곽 받기 (OSM Overpass) | `s01_candidates` · `s03_prior` 주변 건물 |
| 5 | 점을 품은 건물 = 대상 건물 확정 | `s02_target` |
| 6 | 실제 실행 결과(GIS 건물통합정보)와 비교 | `s02` · `s03` 실습 데이터 |

> API 키 없이 끝까지 돈다. 네트워크가 막히면 실습 데이터(`data/ipark/`)로 넘어간다.
> 결과물: `out/footprint.json`, `out/w04_1_footprint.png`

## 1. 준비

In [ ]:
# ── 0. 실습 데이터 확인 ─────────────────────────────────────────
# 이 노트북은 옆에 data/ipark/ 폴더가 있다고 가정한다 (아이파크타워 실제 파이프라인 결과).
import os, sys, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/kloud80/dataminig_cau.git"   # 강의 저장소 (필요한 폴더만 받는다)
REPO_DIR = "중앙대26특강/notebooks/w04_colab"

DATA = Path('data/ipark')
if not DATA.exists() and REPO_URL:
    print('실습 데이터를 내려받는 중...')
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--sparse', REPO_URL, 'w04_repo'], check=False)
    subprocess.run(['git', '-C', 'w04_repo', 'sparse-checkout', 'set', REPO_DIR], check=False)
    hits = sorted(Path('w04_repo').glob('**/data/ipark'))
    if hits:
        os.chdir(hits[0].parent.parent)        # data/ 의 부모 폴더로 이동
if not DATA.exists():
    raise FileNotFoundError(
        'data/ipark/ 폴더가 없습니다.\n'
        '  1) 수업 저장소를 clone 하고 이 노트북을 w04_colab 폴더에서 여세요.\n'
        '  2) Colab 이라면 네트워크를 확인하고 이 셀을 다시 실행하세요.\n'
        '  (또는 data/ipark 폴더를 왼쪽 파일 탭에 그대로 올려도 됩니다)')

OUT = Path('out'); OUT.mkdir(exist_ok=True)
print('작업 폴더', Path.cwd())
print('실습 데이터', sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time, requests
from shapely.geometry import Polygon, Point
from pyproj import Transformer, Geod, Proj

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 한글 폰트 — 그래프 글자가 네모로 깨지지 않게 (Colab 은 나눔고딕을 설치, 윈도는 맑은 고딕)
import platform
import matplotlib.font_manager as fm
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        os.system('apt-get -qq install -y fonts-nanum > /dev/null')
    if os.path.exists(nanum):
        fm.fontManager.addfont(nanum)
        plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

UA = {'User-Agent': 'cau-urban-seminar-w04/1.0 (Colab teaching notebook)'}   # OSM 서버는 누가 부르는지 밝히라고 한다
OFFLINE = False      # True 로 바꾸면 네트워크를 전혀 쓰지 않고 실습 데이터만 쓴다

In [ ]:
# ── API 키 (전부 선택) ──────────────────────────────────────────
# Colab: 왼쪽 열쇠 아이콘(보안 비밀)에 이름 그대로 등록하고 '노트북 액세스'를 켠다.
# 로컬: 환경변수로 넣는다. 키 값은 화면에 절대 찍지 않는다.
def get_secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    v = os.environ.get(name)
    return v.strip() if v else None

_SECRETS = []
def secret(name):
    v = get_secret(name)
    if v:
        _SECRETS.append(v)
    print(f'{name:15s}', '있음' if v else '없음')
    return v

def scrub(e):
    """오류 메시지에 URL 째로 키가 섞여 나오는 일이 있다 → 지우고 보여 준다."""
    s = str(e)
    for v in _SECRETS:
        s = s.replace(v, '***')
    return s[:160]

VWORLD_KEY = secret('VWORLD_KEY')      # 브이월드 오픈API (지오코딩) — 없어도 된다

## 2. 주소 → 좌표 (지오코딩)

### s00 — 주소 정리
사람이 쓴 주소는 지저분하다. 괄호, 줄바꿈, `101동` 같은 동 표기가 섞인다.
파이프라인은 특수문자를 지우고, 끝에 붙은 **숫자·영문 동 표기만** 따로 떼어 둔다 (법정동 `삼성동` 은 남긴다).

In [ ]:
import re

def s00_normalize(raw):
    s = re.sub(r'[\t\r\n]+', ' ', raw)
    s = re.sub(r'[^\w\s\-.,()가-힣]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    m = re.search(r'\s*(제?\s*[0-9A-Za-z]{1,4}\s*동)\s*$', s)
    dong = None
    if m and re.search(r'[0-9A-Za-z]', m.group(1)):
        dong, s = m.group(1).replace(' ', ''), s[:m.start()].strip()
    return s, dong

for t in ['서울특별시  강남구 영동대로 520', '서울 송파구 올림픽로 99 101동', '서울특별시 강남구 삼성동 160']:
    print(f'{t!r:40s} →', s00_normalize(t))

### 지오코더 두 개

| | VWorld (국토교통부) | Nominatim (OpenStreetMap) |
|---|---|---|
| 키 | 필요 (무료 발급) | 없음 |
| 한국 도로명·지번 | **번지까지 정확** | 번지를 **거의 모른다** |
| 제한 | 일 4만 건 | **초당 1건**, User-Agent 필수 |

키가 있으면 VWorld, 없으면 Nominatim 으로 간다.
Nominatim 이 번지를 모르면 **길 전체의 중간점**을 돌려준다 — 그 점은 건물에서 수백 m~수 km 떨어져 있다.
그래서 그때는 **건물 이름**으로 한 번 더 묻는다 (OSM 에 그 이름이 등록돼 있으면 건물을 바로 찾는다).

In [ ]:
ADDRESS = '서울특별시 강남구 영동대로 520'
NAME_HINT = '현대아이파크타워'     # OSM 에 등록된 건물 이름. Nominatim 이 번지를 못 찾을 때만 쓴다

query, dong_hint = s00_normalize(ADDRESS)

# 실제 파이프라인이 이 건물에 쓴 점 (정답 확인용)
s01 = json.load(open(DATA / 's01_candidates.json', encoding='utf-8'))
REF = (s01['geocode']['lng'], s01['geocode']['lat'])
geod = Geod(ellps='WGS84')
def dist_m(a, b):
    return geod.inv(a[0], a[1], b[0], b[1])[2]

def vworld_geocode(addr):
    for kind in ('road', 'parcel'):
        p = {'service': 'address', 'request': 'getcoord', 'version': '2.0', 'crs': 'epsg:4326',
             'address': addr, 'refine': 'true', 'simple': 'false', 'format': 'json', 'type': kind,
             'key': VWORLD_KEY, 'domain': 'http://localhost'}
        r = requests.get('https://api.vworld.kr/req/address', params=p,
                         headers={'Referer': 'http://localhost'}, timeout=15)
        resp = r.json().get('response', {})
        if resp.get('status') == 'OK':
            pt, ref = resp['result']['point'], resp.get('refined', {})
            return {'lng': float(pt['x']), 'lat': float(pt['y']), 'label': ref.get('text', addr),
                    'building': (ref.get('structure') or {}).get('detail'), 'kind': 'building'}
    return None

_last_call = [0.0]
def nominatim(q):
    """OSM Nominatim — 이용 정책: 초당 1건, 누가 부르는지 User-Agent 로 밝힐 것."""
    wait = 1.0 - (time.time() - _last_call[0])
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()
    r = requests.get('https://nominatim.openstreetmap.org/search',
                     params={'q': q, 'format': 'jsonv2', 'limit': 1, 'countrycodes': 'kr'},
                     headers=UA, timeout=15)
    r.raise_for_status()
    js = r.json()
    if not js:
        return None
    x = js[0]
    return {'lng': float(x['lon']), 'lat': float(x['lat']), 'label': x['display_name'][:40],
            'building': x.get('name'), 'kind': x['category']}     # kind: building / highway / place ...

trials, geo = [], None
def attempt(method, q, fn):
    try:
        g = fn(q)
    except Exception as e:
        print(f'{method} 실패: {scrub(e)}')
        return None
    trials.append({'방법': method, '질의': q,
                   '결과 종류': g['kind'] if g else '없음',
                   '이름': (g or {}).get('building') or '',
                   '정답 점까지(m)': round(dist_m((g['lng'], g['lat']), REF), 1) if g else None})
    return g

if not OFFLINE and VWORLD_KEY:
    geo = attempt('VWorld', query, vworld_geocode)
if not OFFLINE and geo is None:
    g = attempt('Nominatim (주소)', query, nominatim)
    if g and g['kind'] == 'building':
        geo = g
    else:
        print('Nominatim 이 번지를 몰라 건물이 아닌 것을 돌려줬다 → 건물 이름으로 다시 묻는다')
        g = attempt('Nominatim (건물명)', NAME_HINT, nominatim)
        if g and g['kind'] == 'building':
            geo = g
if geo is None:
    print('지오코딩 실패 또는 OFFLINE → 실습 데이터의 좌표를 쓴다')
    geo = {'lng': REF[0], 'lat': REF[1], 'label': s01['geocode']['refined'], 'building': None, 'kind': 'staged'}
    trials.append({'방법': '실습 데이터 (s01)', '질의': '-', '결과 종류': 'staged', '이름': '',
                   '정답 점까지(m)': 0.0})
geo['source'] = trials[-1]['방법']

display(pd.DataFrame(trials))
print(f"\n채택: {geo['source']}  →  경도 {geo['lng']:.6f}, 위도 {geo['lat']:.6f}")

**읽는 법** — `정답 점까지(m)` 는 실제 파이프라인(s01)이 쓴 점과의 거리다.
Nominatim 이 주소로 찾은 결과가 `highway` (도로)라면, 그 점은 `영동대로` 라는 **길의 어딘가**다. 건물이 아니다.
공짜 데이터는 공짜인 이유가 있다. 그래도 **이름**으로는 찾는다 — 누군가 OSM 에 그 건물을 그려 넣고 이름을 달아 둔 덕분이다.

## 3. 왜 국소 미터 좌표로 바꾸나

경위도는 **각도**다. 길이가 아니다. 서울에서 경도 0.001° 와 위도 0.001° 는 길이가 다르다.
면적·거리·벽 폭을 재려면 먼저 미터로 바꿔야 한다.

In [ ]:
lng, lat = geo['lng'], geo['lat']
print(f'경도 0.001° = {dist_m((lng, lat), (lng + 0.001, lat)):6.1f} m  (동서)')
print(f'위도 0.001° = {dist_m((lng, lat), (lng, lat + 0.001)):6.1f} m  (남북)')
print('→ 경위도 그대로 면적을 재면 동서 방향이 약 20% 짧게 잡힌다')

### 그럼 한국 표준 좌표계(EPSG:5186)를 쓰면?
EPSG:5186 도 미터다. 그런데 **격자의 위쪽(도북)이 진짜 북쪽(진북)과 조금 어긋난다.**
중부원점(동경 127°)에서 멀어질수록 커진다. 거리 사진의 방위각(진북 기준)과 섞으면 그만큼 벽 방향이 틀어진다.

In [ ]:
p5186 = Proj('EPSG:5186')
for name, (x, y) in {'아이파크타워 (서울)': (lng, lat), '부산시청': (129.0756, 35.1796)}.items():
    print(f'{name:14s} 도북-진북 차이 {p5186.get_factors(x, y).meridian_convergence:5.2f}°')

### LOCAL_ENU — 건물마다 자기 원점
그래서 파이프라인은 **건물마다 따로** 좌표계를 만든다.
원점 = 건물 중심의 경위도. 그 자리에 중심을 맞춘 횡메르카토르(축척 1) 투영이다.

* `+X` = 동쪽, `+Y` = 북쪽, `+Z` = 위, 단위 m
* 원점이 건물 한가운데라 **왜곡이 사실상 0** 이고, 위쪽이 **정확히 진북**이다

아래 `LocalFrame` 은 파이프라인 `core/geo.py` 의 정의를 그대로 옮긴 것이다 (pyproj, `always_xy=True` → 항상 경도·위도 순서).

In [ ]:
class LocalFrame:
    """LOCAL_ENU: 원점 경위도에 맞춘 횡메르카토르(k=1). +X 동, +Y 북, 단위 m."""
    def __init__(self, lng, lat):
        self.origin = (lng, lat)
        proj = (f'+proj=tmerc +lat_0={lat:.10f} +lon_0={lng:.10f} '
                f'+k=1 +x_0=0 +y_0=0 +ellps=GRS80 +units=m +no_defs')
        self._fwd = Transformer.from_crs('EPSG:4326', proj, always_xy=True)
        self._inv = Transformer.from_crs(proj, 'EPSG:4326', always_xy=True)

    def to_local(self, lng, lat):
        return self._fwd.transform(lng, lat)

    def to_lnglat(self, x, y):
        return self._inv.transform(x, y)

    def ring_to_local(self, ring):
        a = np.asarray(ring, float)
        x, y = self.to_local(a[:, 0], a[:, 1])
        return np.column_stack([x, y])

    def ring_to_lnglat(self, ring):
        a = np.asarray(ring, float)
        x, y = self.to_lnglat(a[:, 0], a[:, 1])
        return np.column_stack([x, y])


def signed_area(r):
    x, y = r[:, 0], r[:, 1]
    return 0.5 * float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def to_ccw(ring):
    """닫힌 링이면 마지막 점을 떼고, 반시계방향(CCW)으로 맞춘다 — 입면 법선 계산의 전제."""
    a = np.asarray(ring, float)
    if len(a) > 1 and np.allclose(a[0], a[-1]):
        a = a[:-1]
    return a if signed_area(a) > 0 else a[::-1].copy()

f = LocalFrame(lng, lat)
print('원점 →', np.round(f.to_local(lng, lat), 6))
print('동쪽 0.001° →', np.round(f.to_local(lng + 0.001, lat), 2), 'm')
print('북쪽 0.001° →', np.round(f.to_local(lng, lat + 0.001), 2), 'm')

## 4. 주변 건물 윤곽 받기 — 공공데이터가 막혔을 때

원래 파이프라인의 1순위는 국토교통부 **GIS건물통합정보** (VWorld NED WFS) 다.
건물 외곽선에 대장 속성(높이·층수·용도·사용승인일)까지 붙어 나오는 가장 좋은 데이터다.

그런데 지금 이 서비스는 우리 키로 **막혀 있다** (`INCORRECT_KEY`). 공공데이터는 이런 일이 잦다 — 승인 범위, 서비스 개편, 일시 장애.
**좋은 파이프라인은 1순위가 막혀도 멈추지 않는다.** 파이프라인의 실제 폴백이 OpenStreetMap **Overpass API** 다.

| | GIS건물통합정보 (NED) | OSM Overpass |
|---|---|---|
| 누가 그렸나 | 국가 (건축물대장 연계) | 자원봉사자 |
| 높이·층수 | 대부분 있음 | **드물다** (태그가 있으면 씀) |
| 키 | 필요 | 없음 |
| 지금 상태 | **차단** | 동작 (서버가 느릴 때가 많다) |

Overpass 는 공개 서버가 여러 개다. 하나가 바쁘면(504) 다음 서버로 넘어가고, 다 실패하면 실습 데이터로 넘어간다.

In [ ]:
OVERPASS = ['https://overpass-api.de/api/interpreter',
            'https://maps.mail.ru/osm/tools/overpass/api/interpreter',
            'https://overpass.kumi.systems/api/interpreter']
RADIUS_M = 150                      # 파이프라인 s03 의 주변 건물 반경
FLOOR_H = 3.3                       # 층수만 있을 때 쓰는 기본 층고 (파이프라인 default)
CACHE = OUT / 'osm_overpass.json'   # 한 번 받으면 다시 부르지 않는다 (공용 서버 예의)

def overpass(lat, lng, radius):
    q = f'[out:json][timeout:25];(way["building"](around:{radius},{lat:.6f},{lng:.6f}););out geom tags;'
    for ep in OVERPASS:
        host = ep.split('/')[2]
        try:
            t = time.time()
            r = requests.post(ep, data={'data': q}, headers=UA, timeout=60)
            if r.status_code == 200 and r.text.lstrip().startswith('{'):
                print(f'  {host} → 성공 ({time.time() - t:.0f}초)')
                return r.json()['elements']
            print(f'  {host} → HTTP {r.status_code}')
        except requests.RequestException as e:
            print(f'  {host} → {type(e).__name__}')
    return None

def _num(v):
    try:
        return float(str(v).lower().replace('m', '').strip())
    except (TypeError, ValueError):
        return None

def parse_osm(elements):
    out = []
    for el in elements:
        geom = el.get('geometry') or []
        if len(geom) < 4:
            continue
        tags = el.get('tags', {})
        out.append({'id': f"osm_{el['id']}", 'name': tags.get('name'),
                    'ring': [[g['lon'], g['lat']] for g in geom],
                    'height': _num(tags.get('height')), 'levels': _num(tags.get('building:levels'))})
    return out

def staged_buildings():
    """실습 데이터 s03 의 주변 건물 = 실제 실행 때 Overpass 에서 받은 것 (국소 좌표 → 경위도로 되돌린다)."""
    s03 = json.load(open(DATA / 's03_prior.json', encoding='utf-8'))
    fr = LocalFrame(*s03['origin_lnglat'])
    out = []
    for n in s03['neighbors']:
        ring = fr.ring_to_lnglat(n['footprint_local']).tolist()
        src = n['height_source']
        out.append({'id': n['building_id'], 'name': None, 'ring': ring + ring[:1],
                    'height': n['height_m'] if src == 'osm_height' else None,
                    'levels': None})
    return out

key = [round(geo['lat'], 6), round(geo['lng'], 6), RADIUS_M]
elements, osm_source = None, None
if CACHE.exists():
    c = json.load(open(CACHE, encoding='utf-8'))
    if c.get('key') == key:
        elements, osm_source = c['elements'], 'overpass (저장본)'
if elements is None and not OFFLINE:
    print('Overpass 에 묻는 중... (보통 10~40초, 서버가 바쁘면 최대 3분 뒤 실습 데이터로 넘어간다)')
    elements = overpass(geo['lat'], geo['lng'], RADIUS_M)
    if elements is not None:
        osm_source = 'overpass'
        json.dump({'key': key, 'elements': elements}, open(CACHE, 'w', encoding='utf-8'), ensure_ascii=False)

if elements is not None:
    buildings = parse_osm(elements)
else:
    print('Overpass 실패 또는 OFFLINE → 실습 데이터(s03 주변 건물)로 진행')
    buildings, osm_source = staged_buildings(), 'staged_s03'
print(f'건물 {len(buildings)}동  (출처: {osm_source})')
print('높이 태그가 있는 건물:', sum(b['height'] is not None for b in buildings), '동',
      '/ 층수 태그:', sum(b['levels'] is not None for b in buildings), '동')

## 5. 대상 건물 고르기 (s01 후보 → s02 확정)

파이프라인의 규칙을 그대로 쓴다.

1. 지오코딩 점에서 **30 m 안**의 건물을 후보로 모은다. 15 ㎡ 미만 조각은 버린다.
2. 점수: 점을 **품으면 1.0**, 아니면 `1 / (1 + 거리)`. 30 ㎡ 미만(경비실·창고)은 −0.6.
3. 1등이 대상 건물. (실제 서비스에서는 사람이 지도에서 클릭해 확정한다)

In [ ]:
gframe = LocalFrame(geo['lng'], geo['lat'])         # 지오코딩 점이 원점
pt = Point(0, 0)
for b in buildings:
    P = Polygon(gframe.ring_to_local(b['ring'])).buffer(0)
    b['area_m2'] = P.area
    b['dist_m'] = P.distance(pt)
    b['contains'] = P.contains(pt)
    b['score'] = (1.0 if b['contains'] else 1 / (1 + b['dist_m'])) - (0.6 if P.area < 30 else 0)

cands = sorted([b for b in buildings if b['dist_m'] <= 30 and b['area_m2'] >= 15], key=lambda b: -b['score'])
if not cands:                                       # 30 m 안에 없으면 가장 가까운 건물
    cands = sorted(buildings, key=lambda b: b['dist_m'])[:1]
target = cands[0]

display(pd.DataFrame([{'id': b['id'], '이름': b['name'], '면적(㎡)': round(b['area_m2']),
                       '거리(m)': round(b['dist_m'], 1), '점 포함': b['contains'],
                       '점수': round(b['score'], 3)} for b in cands[:5]]))
print('대상 건물:', target['id'], target['name'] or '')

### 대상 건물의 LOCAL_ENU
이제 원점을 **건물 윤곽의 중심**으로 옮긴다 (파이프라인 s03 과 같은 정의 — 경위도 다각형의 중심).
반경 150 m 안의 다른 건물은 **주변 건물**로 둔다. 나중에 맞벽 판정과 사진의 가림 계산에 쓴다.
OSM 에 높이가 없으면 `층수 × 3.3 m`, 그것도 없으면 9 m 로 둔다 (파이프라인과 같은 규칙).

In [ ]:
cen = Polygon(target['ring']).centroid
frame = LocalFrame(cen.x, cen.y)
fp = to_ccw(frame.ring_to_local(target['ring']))
T = Polygon(fp)

neighbors = []
for b in buildings:
    if b is target:
        continue
    ring = to_ccw(frame.ring_to_local(b['ring']))
    if len(ring) < 3:
        continue
    P = Polygon(ring).buffer(0)
    d = P.distance(T)
    if d > RADIUS_M:
        continue
    h, src = ((b['height'], 'osm_height') if b['height'] else
              (b['levels'] * FLOOR_H, 'osm_levels') if b['levels'] else (9.0, 'default'))
    neighbors.append({'building_id': b['id'], 'name': b['name'], 'height_m': round(h, 1), 'height_source': src,
                      'distance_m': round(d, 1), 'overlap_m2': round(P.intersection(T).area, 1),
                      'footprint_local': ring.round(3).tolist()})

print(f'대상 건물 면적 {T.area:,.1f} ㎡ · 꼭짓점 {len(fp)}개 · 둘레 {T.length:.1f} m')
print(f'원점 (경도, 위도) = ({cen.x:.7f}, {cen.y:.7f})')
print(f'주변 건물 {len(neighbors)}동 (반경 {RADIUS_M} m)')
display(pd.DataFrame(neighbors).drop(columns='footprint_local')
        .sort_values('distance_m').head(8))

## 6. 그려 보기

In [ ]:
from matplotlib.patches import Polygon as MplPoly, Circle
from matplotlib import cm, colors

fig, ax = plt.subplots(figsize=(11, 9))
norm = colors.Normalize(0, max([n['height_m'] for n in neighbors] + [30]))
for n in neighbors:
    r = np.asarray(n['footprint_local'])
    ax.add_patch(MplPoly(r, closed=True, fc=cm.Blues(0.25 + 0.7 * norm(n['height_m'])), ec='#444', lw=0.6))
    if n['name']:
        c = r.mean(0); ax.text(c[0], c[1], n['name'], fontsize=9, ha='center', color='#222')
ax.add_patch(MplPoly(fp, closed=True, fc='#E8452C', ec='k', lw=1.5, alpha=0.9))
ax.annotate(target['name'] or '대상 건물', xy=(0, 0), xytext=(30, -30), fontsize=13, weight='bold',
            color='#B0281A', arrowprops=dict(arrowstyle='-', color='#B0281A'))
ax.add_patch(Circle((0, 0), RADIUS_M, fill=False, ls='--', ec='gray'))
gx, gy = frame.to_local(geo['lng'], geo['lat'])
ax.plot(gx, gy, marker='*', ms=18, mfc='gold', mec='k', ls='', label=f"지오코딩 점 ({geo['source']})")
ax.annotate('N', xy=(RADIUS_M * 0.9, RADIUS_M * 0.95), xytext=(RADIUS_M * 0.9, RADIUS_M * 0.75),
            ha='center', fontsize=14, weight='bold', arrowprops=dict(arrowstyle='-|>', lw=2))
ax.set_xlim(-RADIUS_M * 1.05, RADIUS_M * 1.05); ax.set_ylim(-RADIUS_M * 1.05, RADIUS_M * 1.05)
ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.set_xlabel('X = 동쪽 (m)'); ax.set_ylabel('Y = 북쪽 (m)')
fig.colorbar(cm.ScalarMappable(norm=norm, cmap='Blues'), ax=ax, shrink=0.7, label='주변 건물 높이 (m)')
ax.legend(loc='lower left')
ax.set_title(f'LOCAL_ENU — 대상 건물과 반경 {RADIUS_M} m 주변 건물 (OSM)', fontsize=14)
plt.tight_layout(); plt.savefig(OUT / 'w04_1_footprint.png', dpi=150); plt.show()

### (선택) 웹 지도로 보기
folium 은 **위도, 경도** 순서를 쓴다. pyproj(`always_xy`)·GeoJSON 은 **경도, 위도** 순서다.
이 순서를 헷갈리면 건물이 남극 근처에 찍힌다 — 좌표 작업의 가장 흔한 실수다.

In [ ]:
try:
    import folium
    m = folium.Map(location=[cen.y, cen.x], zoom_start=17, tiles='OpenStreetMap')
    for n in neighbors:
        ll = frame.ring_to_lnglat(n['footprint_local'])
        folium.Polygon([(la, lo) for lo, la in ll], color='#2C6FB7', weight=1, fill=True, fill_opacity=0.3,
                       tooltip=f"{n['name'] or n['building_id']} · {n['height_m']} m").add_to(m)
    folium.Polygon([(la, lo) for lo, la in target['ring']], color='#E8452C', weight=3, fill=True,
                   fill_opacity=0.5, tooltip=target['name'] or '대상').add_to(m)
    folium.Marker([geo['lat'], geo['lng']], tooltip='지오코딩 점').add_to(m)
    display(m)
except ImportError:
    print('folium 이 없어 지도를 건너뜀 (pip install folium)')

## 7. 실제 실행 결과와 비교 — 솔직하게

실습 데이터(`s02_target.json`, `s03_prior.json`)는 파이프라인이 이 건물을 **실제로 돌린 결과**다.
그때는 GIS건물통합정보의 외곽선(국가 데이터)을 받을 수 있었다. 지금 우리 결과(OSM)와 나란히 놓는다.

### 먼저 — 좌표 변환이 파이프라인과 똑같은가
GIS 외곽선을 우리 `LocalFrame` 으로 바꿔 실습 데이터의 `footprint_local` 과 비교한다. 같은 코드면 차이가 0 이어야 한다.

In [ ]:
s02 = json.load(open(DATA / 's02_target.json', encoding='utf-8'))
s03 = json.load(open(DATA / 's03_prior.json', encoding='utf-8'))
gis_ring = s02['candidate']['polygon_wgs84'][0]

c0 = Polygon(gis_ring).centroid
sframe = LocalFrame(c0.x, c0.y)
mine = to_ccw(sframe.ring_to_local(gis_ring))
ref = np.asarray(s03['footprint_local'])
print('원점 차이 (도):', np.abs(np.array([c0.x, c0.y]) - s03['origin_lnglat']).max())
print(f'꼭짓점 최대 차이: {np.abs(mine - ref).max() * 1000:.6f} mm  ← 파이프라인과 같은 좌표계')

### OSM 윤곽 vs 국가 GIS 윤곽
두 윤곽을 **같은 좌표계**(실습 데이터의 원점)에 올리고, 같은 범위의 위성 정사영상(VWorld) 위에 겹친다.

In [ ]:
gis = Polygon(ref)
osm_local = to_ccw(sframe.ring_to_local(target['ring']))
osm = Polygon(osm_local).buffer(0)
inter, union = gis.intersection(osm).area, gis.union(osm).area
c = s02['candidate']; led = s03.get('ledger') or {}

cmp = pd.DataFrame([
    {'출처': '국가 GIS건물통합정보 (실제 실행)', '꼭짓점': len(ref), '면적(㎡)': round(gis.area, 1)},
    {'출처': f'OSM ({osm_source})', '꼭짓점': len(osm_local), '면적(㎡)': round(osm.area, 1)},
    {'출처': '건축물대장 건축면적 (참고)', '꼭짓점': None, '면적(㎡)': led.get('archArea')},
])
cmp['꼭짓점'] = cmp['꼭짓점'].astype('Int64')
display(cmp)
print(f'겹친 면적 {inter:,.0f} ㎡ · IoU {inter / union:.2f} · 중심 차이 '
      f'{np.hypot(*(np.array(osm.centroid.coords[0]) - np.array(gis.centroid.coords[0]))):.1f} m')

from PIL import Image
img = Image.open(DATA / 'ortho_local.png')
x0, y0, x1, y1 = s03['ortho_bounds_local']
fig, ax = plt.subplots(figsize=(9, 10))
ax.imshow(img, extent=[x0, x1, y0, y1])
ax.add_patch(MplPoly(ref, closed=True, fill=False, ec='#FF3B30', lw=3, label='국가 GIS (s02/s03)'))
ax.add_patch(MplPoly(osm_local, closed=True, fill=False, ec='#00D1FF', lw=3, ls='--', label='OSM (오늘)'))
for n in s03['neighbors']:
    r = np.asarray(n['footprint_local'])
    if Polygon(r).buffer(0).distance(gis) < 3:
        ax.add_patch(MplPoly(r, closed=True, fill=False, ec='yellow', lw=1.2, ls=':'))
ax.set_xlim(-60, 60); ax.set_ylim(-70, 70); ax.set_aspect('equal')
ax.legend(loc='lower left', fontsize=12)
ax.set_title('같은 건물, 두 개의 윤곽 (배경: VWorld 위성, 노란 점선: 붙어 있는 OSM 건물)', fontsize=12)
plt.tight_layout(); plt.savefig(OUT / 'w04_1_gis_vs_osm.png', dpi=150); plt.show()

**무엇이 다른가** (아이파크타워 기준, 2026년 10월 OSM)

* OSM 의 `현대아이파크타워` 는 **꼭짓점 4개짜리 가는 사각형**이고 면적이 국가 GIS 의 **절반 정도**다.
  누군가 위성사진을 보고 손으로 그렸을 것이다. 바로 옆에 `현대아이파크타워 별관` 이 **따로** 그려져 있다.
* 위성사진은 바로 위에서 찍은 것이 아니다. 높은 건물일수록 **지붕이 옆으로 밀려 보인다**(기복변위).
  사진만 보고 그리면 바닥이 아니라 지붕을 따라 그리기 쉽다 — 위 그림에서 직접 확인해 보자.
* 국가 GIS 윤곽은 꼭짓점 11개, 면적은 대장의 건축면적과 거의 같다 (둘 다 같은 대장에서 왔다).
* OSM 의 높이 94 m · 15층 태그는 대장 값과 같다 — **독립된 근거가 아니라 대장을 옮겨 적은 것**일 가능성이 크다.

> **교훈** — 폴백은 "없는 것보다 낫다" 이지 "같다" 가 아니다. 폴백으로 만든 결과에는 **출처를 꼭 남긴다.**
> 아래 저장 파일의 `source` 필드가 그 역할을 한다.

### 그리고 이 차이가 실제 실행에서 사고를 냈다
실제 실행에서 국가 GIS 윤곽은 대상, OSM 은 주변 건물로 쓰였다. 그런데 OSM 의 아이파크타워 자신이 **주변 건물 목록에 남았다.**
파이프라인은 "대상과 50% 넘게 겹치는 주변 건물은 같은 건물로 보고 뺀다" 고 했는데, 이 OSM 사각형은 대상 면적의 **45%** 만 겹쳤기 때문이다.

In [ ]:
for n in s03['neighbors']:
    P = Polygon(n['footprint_local']).buffer(0)
    ov = P.intersection(gis).area
    if ov > 1:
        print(f"{n['building_id']}: 높이 {n['height_m']} m, 대상과 겹친 면적 {ov:,.0f} ㎡ "
              f"= 대상의 {ov / gis.area:.0%}  → 50% 기준 미달이라 '주변 건물'로 남음")
print('→ 3번 노트북에서 이것이 입면의 맞벽 판정을 어떻게 틀리게 만드는지 본다')

## 8. 저장 — `out/footprint.json`

In [ ]:
result = {
    'address': ADDRESS,
    'geocode': {'lng': geo['lng'], 'lat': geo['lat'], 'source': geo['source']},
    'building_id': target['id'], 'name': target['name'],
    'source': osm_source,                         # 폴백임을 남긴다
    'crs': 'LOCAL_ENU', 'origin_lnglat': [cen.x, cen.y],
    'footprint_local': fp.round(3).tolist(),
    'footprint_wgs84': [list(map(float, p)) for p in target['ring']],
    'area_m2': round(T.area, 1),
    'osm_height_m': target['height'], 'osm_levels': target['levels'],
    'neighbors': neighbors,
    'compare_gis': {'gis_area_m2': round(gis.area, 1), 'iou': round(inter / union, 3)},
}
json.dump(result, open(OUT / 'footprint.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

print('── W04-1 요약 ───────────────────────────')
print(f"주소        {ADDRESS}")
print(f"지오코딩    {geo['source']}  ({geo['lng']:.6f}, {geo['lat']:.6f})")
print(f"대상 건물   {target['name'] or target['id']}  (윤곽 출처: {osm_source})")
print(f"외곽선      꼭짓점 {len(fp)}개 · 면적 {T.area:,.1f} ㎡")
print(f"주변 건물   {len(neighbors)}동 / 반경 {RADIUS_M} m")
print(f"국가 GIS 와 IoU {inter / union:.2f}  (GIS 면적 {gis.area:,.1f} ㎡)")
print('저장       ', ', '.join(sorted(p.name for p in OUT.glob('*'))))

## 해 볼 것

1. `ADDRESS` 를 자기 학교나 집 주소로 바꿔 보자. Nominatim 이 번지를 찾는가, 길을 찾는가? `NAME_HINT` 에 건물 이름을 넣으면 달라지는가?
2. 6번 그림에서 높이가 **9 m (기본값)** 로 찍힌 건물은 몇 동인가? 이 숫자로 "주변 건물이 사진을 가린다" 를 계산하면 무엇이 틀릴까?
3. 3번에서 EPSG:5186 의 도북-진북 차이를 강릉·제주에서도 재 보자. 거리 사진 방위각과 섞으면 20 m 떨어진 벽에서 몇 cm 어긋나는가?
4. 7번의 "50% 넘게 겹치면 같은 건물" 규칙을 어떻게 고치면 오늘 같은 사고를 막을까? 그 규칙이 **다른 건물을 잘못 지우는** 경우는 없을까?
5. OSM 은 누구나 고칠 수 있다. 아이파크타워의 OSM 윤곽이 틀렸다면, 고치는 것이 맞을까? 고친다면 무엇을 근거로?

### 참고
* addr2blend 파이프라인 (사내 연구용) — `core/geo.py`, `sources/osm.py`, `stages/s00_s02_target.py`
* VWorld 오픈API 지오코더 — `api.vworld.kr/req/address`
* Nominatim 이용 정책 — operations.osmfoundation.org/policies/nominatim (초당 1건, User-Agent 필수)
* Overpass API — wiki.openstreetmap.org/wiki/Overpass_API
* 지도 데이터 © OpenStreetMap contributors (ODbL)